# RISC-V Pipelined Processor - PYNQ-Z2 Verification

| Block | AXI Base | Purpose |
|---|---|---|
| `bram_controller_iram` | `0x42000000` | Instruction BRAM (4096 x 32-bit) |
| `bram_controller_dram` | `0x40000000` | Data BRAM (4096 x 32-bit) |
| `axi_gpio_reset` | `0x41200000` | 1-bit GPIO -> core resetn |

GPIO=0 -> core halted, GPIO=1 -> core running

In [1]:
# =============================================================
# Load Overlay & Map Hardware
# =============================================================
from pynq import Overlay, MMIO
import time
import numpy as np

BITSTREAM = "riscv_pynq_lfg.bit"  #

print("Loading Overlay...")
overlay = Overlay(BITSTREAM)
print("Overlay Loaded.")

print("IPs:", list(overlay.ip_dict.keys()))
print("Mem:", list(overlay.mem_dict.keys()))

# BRAM controllers are in mem_dict
iram_info = overlay.mem_dict['bram_controller_iram']
dram_info = overlay.mem_dict['bram_controller_dram']
iram = MMIO(iram_info['phys_addr'], iram_info['addr_range'])
dram = MMIO(dram_info['phys_addr'], dram_info['addr_range'])

# GPIO is in ip_dict
gpio_info = overlay.ip_dict['axi_gpio_reset']
gpio = MMIO(gpio_info['phys_addr'], gpio_info['addr_range'])

def halt_core():
    gpio.write(0x0, 0x0)

def run_core():
    gpio.write(0x0, 0x1)

# for debugging
print(f"IRAM: {iram_info['addr_range']//4} words, DRAM: {dram_info['addr_range']//4} words")
print("Hardware mapped.")

Loading Overlay...


Overlay Loaded.
IPs: ['axi_gpio_reset', 'processing_system7_0']
Mem: ['bram_controller_dram', 'bram_controller_iram', 'PSDDR']
IRAM: 4096 words, DRAM: 4096 words
Hardware mapped.


In [2]:
# =============================================================
# Memory Sanity Check
# =============================================================
print("=" * 50)
print("SANITY CHECK")
print("=" * 50)
halt_core()
errors = 0

for pattern in [0xDEADBEEF, 0x01020304, 0xFFFFFFFF, 0x00000000]:
    iram.write(0x0, pattern)
    rb = iram.read(0x0)
    ok = rb == pattern
    if not ok: errors += 1
    print(f"  IRAM 0x{pattern:08X} -> 0x{rb:08X} [{'OK' if ok else 'FAIL'}]")

for pattern in [0xCAFEBABE, 0x12345678, 0x00000000]:
    dram.write(0x0, pattern)
    rb = dram.read(0x0)
    ok = rb == pattern
    if not ok: errors += 1
    print(f"  DRAM 0x{pattern:08X} -> 0x{rb:08X} [{'OK' if ok else 'FAIL'}]")

# Test multiple offsets
for off in [0x0, 0x4, 0x10, 0x7C, 0x100]:
    dram.write(off, 0xAA000000 | off)
    rb = dram.read(off)
    ok = rb == (0xAA000000 | off)
    if not ok: errors += 1
    print(f"  DRAM[0x{off:03X}] = 0x{rb:08X} [{'OK' if ok else 'FAIL'}]")

gpio.write(0x0, 0x0)
v0 = gpio.read(0x0) & 0x1
gpio.write(0x0, 0x1)
v1 = gpio.read(0x0) & 0x1
gpio_ok = (v0 == 0 and v1 == 1)
if not gpio_ok: errors += 1
print(f"  GPIO: 0->{v0}, 1->{v1} [{'OK' if gpio_ok else 'FAIL'}]")
halt_core()

print("=" * 50)
print("ALL PASSED" if errors == 0 else f"FAILED: {errors} error(s)")
print("=" * 50)

SANITY CHECK
  IRAM 0xDEADBEEF -> 0xDEADBEEF [OK]
  IRAM 0x01020304 -> 0x01020304 [OK]
  IRAM 0xFFFFFFFF -> 0xFFFFFFFF [OK]
  IRAM 0x00000000 -> 0x00000000 [OK]
  DRAM 0xCAFEBABE -> 0xCAFEBABE [OK]
  DRAM 0x12345678 -> 0x12345678 [OK]
  DRAM 0x00000000 -> 0x00000000 [OK]
  DRAM[0x000] = 0xAA000000 [OK]
  DRAM[0x004] = 0xAA000004 [OK]
  DRAM[0x010] = 0xAA000010 [OK]
  DRAM[0x07C] = 0xAA00007C [OK]
  DRAM[0x100] = 0xAA000100 [OK]
  GPIO: 0->0, 1->1 [OK]
ALL PASSED


In [3]:
# =============================================================
# Smoke Test (check if core showss lifesigns)
# =============================================================
print("=" * 50)
print("SMOKE TEST")
print("=" * 50)
halt_core()

# Clear memories
for i in range(16):
    iram.write(i * 4, 0x00000013)  # NOP
    dram.write(i * 4, 0xFFFFFFFF)  # sentinel

# addi x1,x0,42; sw x1,0(x0); addi x2,x0,7; sw x2,4(x0); beq x0,x0,0
smoke = [0x02A00093, 0x00102023, 0x00700113, 0x00202223, 0x00000063]
for i, instr in enumerate(smoke):
    iram.write(i * 4, instr)

run_core()
time.sleep(0.5)
halt_core()

v0 = dram.read(0x0)
v1 = dram.read(0x4)
print(f"DRAM[0] = {v0} (expect 42), DRAM[1] = {v1} (expect 7)")
if v0 == 42 and v1 == 7:
    print(">>> SMOKE TEST PASSED <<<")
else:
    print(">>> SMOKE TEST FAILED <<<")

SMOKE TEST
DRAM[0] = 42 (expect 42), DRAM[1] = 7 (expect 7)
>>> SMOKE TEST PASSED <<<


In [5]:
# =============================================================
# 32-Element Sort Test
# =============================================================
from typing import List, Tuple

def _mask(x, bits): return x & ((1 << bits) - 1)

def enc_lui(rd: int, imm20: int) -> int:
    return (_mask(imm20, 20) << 12) | (_mask(rd, 5) << 7) | 0x37  # lui rd, imm20

def enc_addi(rd: int, rs1: int, imm: int) -> int:
    imm12 = _mask(imm, 12)
    return (imm12 << 20) | (_mask(rs1, 5) << 15) | (0b000 << 12) | (_mask(rd, 5) << 7) | 0x13  # addi

def enc_lw(rd: int, rs1: int, imm: int) -> int:
    imm12 = _mask(imm, 12)
    return (imm12 << 20) | (_mask(rs1, 5) << 15) | (0b010 << 12) | (_mask(rd, 5) << 7) | 0x03  # lw

def enc_sw(rs2: int, rs1: int, imm: int) -> int:
    imm12 = _mask(imm, 12)
    imm_hi = (imm12 >> 5) & 0x7F
    imm_lo = imm12 & 0x1F
    return (imm_hi << 25) | (_mask(rs2, 5) << 20) | (_mask(rs1, 5) << 15) | (0b010 << 12) | (imm_lo << 7) | 0x23  # sw

def enc_slt(rd: int, rs1: int, rs2: int) -> int:
    return (0b0000000 << 25) | (_mask(rs2, 5) << 20) | (_mask(rs1, 5) << 15) | (0b010 << 12) | (_mask(rd, 5) << 7) | 0x33  # slt

def enc_beq(rs1: int, rs2: int, imm: int) -> int:
    # imm is byte offset, must be multiple of 2
    assert imm % 2 == 0
    imm13 = _mask(imm, 13)
    b12   = (imm13 >> 12) & 0x1
    b10_5 = (imm13 >> 5)  & 0x3F
    b4_1  = (imm13 >> 1)  & 0xF
    b11   = (imm13 >> 11) & 0x1
    return (b12 << 31) | (b10_5 << 25) | (_mask(rs2, 5) << 20) | (_mask(rs1, 5) << 15) | (0b000 << 12) | (b4_1 << 8) | (b11 << 7) | 0x63  # beq

def gen_sort32_adjacent_network() -> Tuple[List[int], List[str]]:
    # regs: x7=a, x8=b, x9=t, x12=flag
    x0, x7, x8, x9, x12 = 0, 7, 8, 9, 12
    prog: List[int] = []
    asm:  List[str] = []

    # bubble-sort passes (adjacent compare-swap)
    for p in range(31):
        for j in range(31 - p):
            off = 4 * j
            prog.append(enc_lw(x7, x0, off));      asm.append(f"lw   x7, {off}(x0)")
            prog.append(enc_lw(x8, x0, off + 4));  asm.append(f"lw   x8, {off+4}(x0)")
            prog.append(enc_slt(x9, x8, x7));      asm.append(f"slt  x9, x8, x7")
            prog.append(enc_beq(x9, x0, 12));      asm.append(f"beq  x9, x0, +12")
            prog.append(enc_sw(x8, x0, off));      asm.append(f"sw   x8, {off}(x0)")
            prog.append(enc_sw(x7, x0, off + 4));  asm.append(f"sw   x7, {off+4}(x0)")

    # done flag (@ 0x100)
    prog.append(enc_lui(x12, 0xDEADC));           asm.append("lui  x12, 0xDEADC")
    prog.append(enc_addi(x12, x12, -337));        asm.append("addi x12, x12, -337")
    prog.append(enc_sw(x12, x0, 0x100));          asm.append("sw   x12, 256(x0)")
    prog.append(enc_beq(x0, x0, 0));              asm.append("beq  x0, x0, 0")

    return prog, asm

# Load the program and RISCV-asm translation
PROGRAM, ASM = gen_sort32_adjacent_network()

STATUS_BYTE = 0x100
DONE_FLAG = 0xDEADBEAF
TIMEOUT = 5

np.random.seed(67)
test_data = np.random.randint(-100, 100, size=32).tolist()
golden = sorted(test_data)

print("=" * 60)
print("SORT TEST (32 signed integers)")
print(f"Program: {len(PROGRAM)} instructions (generated; allowed subset only)")
print("=" * 60)

halt_core()
print("[1] Core halted.")

# Clearing IRAM
for i in range(1024):
    iram.write(i * 4, 0x00000013)  # addi x0,x0,0 (riscv NOP)

# Load program into instruction memory
for i, instr in enumerate(PROGRAM):
    iram.write(i * 4, instr)
print(f"[2] Loaded {len(PROGRAM)} instructions into IRAM.")

# Verify the IRAM contents
iram_ok = True
for i, instr in enumerate(PROGRAM):
    v = iram.read(i * 4)
    if v != instr:
        print(f"    IRAM[{i}] MISMATCH: wrote 0x{instr:08X}, read 0x{v:08X}  # {ASM[i]}")
        iram_ok = False
if iram_ok:
    print(f"    All {len(PROGRAM)} instructions verified OK.")
else:
    print("    !!! IRAM VERIFICATION FAILED - DO NOT PROCEED !!!")

# Load test data into DRAM
for i, val in enumerate(test_data):
    dram.write(i * 4, val & 0xFFFFFFFF)
dram.write(STATUS_BYTE, 0x00000000)
print(f"[3] Loaded {len(test_data)} data words + cleared status flag.")

# Verify DRAM writes
data_ok = True
for i, val in enumerate(test_data):
    v = dram.read(i * 4)
    expected = val & 0xFFFFFFFF
    if v != expected:
        sv = v if v < 0x80000000 else v - 0x100000000
        print(f"    DRAM[{i}] MISMATCH: wrote {val}, read {sv} (0x{v:08X})")
        data_ok = False
if data_ok:
    print(f"    All {len(test_data)} data words verified OK.")
else:
    print("    !!! DATA VERIFICATION FAILED - DO NOT PROCEED !!!")

print("[4] Releasing reset...")
run_core()

# Poll for completion
start = time.time()
done = False
while (time.time() - start) < TIMEOUT:
    flag = dram.read(STATUS_BYTE)
    if flag == DONE_FLAG:
        done = True
        break
    time.sleep(0.001)

elapsed = time.time() - start
halt_core()
print(f"[5] Flag: 0x{dram.read(STATUS_BYTE):08X} ({'DONE' if done else 'NOT SET'}) after {elapsed:.4f}s")

# Read & verify results
print()
result = []
for i in range(32):
    v = dram.read(i * 4)
    sv = v if v < 0x80000000 else v - 0x100000000
    result.append(sv)

print("Output:  ", result)
print("Expected:", golden)

errors = 0
for i in range(32):
    if result[i] != golden[i]:
        print(f"  MISMATCH [{i}]: got {result[i]}, expected {golden[i]}")
        errors += 1

print()
print("=" * 60)
if done and errors == 0:
    print("  ALL 32 ELEMENTS SORTED CORRECTLY :D")
elif not done:
    print("  FAIL: Core did not write completion flag D:")
else:
    print(f"  FAIL: {errors} element(s) wrong.")
print("=" * 60)

BUBBLE SORT TEST (32 signed integers)
Program: 27 instructions (embedded, no file needed)
[1] Core halted.
[2] Loaded 27 instructions into IRAM.
    All 27 instructions verified OK.
[3] Loaded 32 data words + cleared status flag.
    All 32 data words verified OK.
[4] Releasing reset...
[5] Flag: 0xDEADBEAF (DONE) after 0.0016s

Output:   [-280, -279, -242, -229, -213, -201, -198, -194, -179, -170, -140, -109, -86, -48, -30, -24, 8, 13, 30, 43, 72, 85, 113, 135, 158, 159, 166, 174, 175, 191, 210, 260]
Expected: [-280, -279, -242, -229, -213, -201, -198, -194, -179, -170, -140, -109, -86, -48, -30, -24, 8, 13, 30, 43, 72, 85, 113, 135, 158, 159, 166, 174, 175, 191, 210, 260]

  >>> ALL 32 ELEMENTS SORTED CORRECTLY <<<


In [ ]:
# =============================================================
# Debug Dump
# =============================================================
halt_core()

print("IRAM (first 30 words)")
for i in range(30):
    v = iram.read(i * 4)
    print(f"  [{i:2d}] 0x{i*4:03X}: 0x{v:08X}")

print("\nDRAM (first 40 words)")
for i in range(40):
    v = dram.read(i * 4)
    sv = v if v < 0x80000000 else v - 0x100000000
    print(f"  [{i:2d}] 0x{i*4:03X}: 0x{v:08X} ({sv})")

print(f"\nStatus flag (0x100)")
print(f"  0x{dram.read(0x100):08X}")